# Week 9 — Work with an LLM assistant

**Research task:** Review an unfamiliar mortgage-prediction analysis as a consultant. After the first model reply, choose whether to ask another question, consult the data-science team or write the client note. The exercise adapts the statistical task in Wiles et al. (2026).

**Python introduced:** the first student-edited function, parameters, local variables, `return`, calling a function twice and an `if/elif` branch controlled by a worker choice.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session09/session09_llms_at_work.ipynb)

Run the setup cell below before anything else. Colab uses OpenRouter. Local JupyterLab uses Ollama by default.

## Prepare the notebook environment

The next cell installs missing SDKs in Colab. Locally it checks that Jupyter is using the course environment and prints the exact repair command if necessary.

In [ ]:
SESSION = "session09"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "A course SDK is missing. In the complete course folder run 'uv sync --frozen', "
        "then run this task with 'uv run python'."
    ) from error

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError("Run this file from inside the complete GenAI_Soc2026 folder.")

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

## Store the exact research material

This cell creates a dictionary of model results, the AI-written interpretation, the coaching instruction and a schema describing the six fields the model must return. Read the results and identify the questionable claims before running it.

In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # local default
model_results = {
    "task": "Predict whether a couple will take out a mortgage",
    "test_cases": 200,
    "actual_mortgage_cases": 80,
    "model_accuracy": 0.80,
    "always_predict_no_mortgage_accuracy": 0.60,
    "false_negatives": 20,
    "evidence": "Performance on held-out historical applications",
}
ai_analysis = (
    "The model is 80% accurate, so it is ready to guide an investment strategy. "
    "Its score shows that increasing a couple's score causes them to take out a mortgage. "
    "It also performs much better than the current baseline, which is no better than chance."
)
assistant_instruction = (
    "Act as a coach. Check the analysis and explain the problems you find. "
    "Ask the consultant questions, but do not write the final client advice."
)
output_schema = {
    "type": "object",
    "properties": {
        "baseline_comparison": {
            "type": "string",
            "enum": ["model_outperforms_baseline", "no_difference", "insufficient_information"],
        },
        "causal_claim": {"type": "string", "enum": ["supported", "not_supported"]},
        "decision_status": {"type": "string", "enum": ["ready", "needs_more_information"]},
        "explanation": {"type": "array", "items": {"type": "string"}},
        "questions_for_worker": {"type": "array", "items": {"type": "string"}},
        "data_science_escalation": {"type": "string"},
    },
    "required": [
        "baseline_comparison", "causal_claim", "decision_status", "explanation",
        "questions_for_worker", "data_science_escalation",
    ],
    "additionalProperties": False,
}

## Define the function

The function receives two values: the current message list and the selected route. It makes one visible route-specific call, prints the raw return, parses it and returns one reply-record dictionary. We use the same function again only if the worker asks another question.

In [ ]:
def call_review_assistant(messages, route):
    """Send the current conversation to one model route and preserve its return."""
    print("Function input — messages:", type(messages), "with", len(messages), "items")
    print("Function input — route:", route, type(route))
    print("Latest message:\n", messages[-1]["content"])

    if route == "openrouter":
        api_key = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter course key (hidden): ")
        response_format = {"type": "json_schema", "json_schema": {
            "name": "work_review", "strict": True,
            "schema": output_schema,
        }}
        with OpenRouter(api_key=api_key) as client:
            response = client.chat.send(
                model=HOSTED_MODEL,
                messages=messages,
                temperature=0,
                response_format=response_format,
            )
        raw_output = response.choices[0].message.content
        model_used = HOSTED_MODEL
    elif route == "ollama":
        response = ollama.chat(think=False,
            model=LOCAL_MODEL,
            messages=messages,
            format=output_schema,
            options={"temperature": 0},
        )
        raw_output = response.message.content
        model_used = LOCAL_MODEL
    else:
        raise ValueError("route must be 'openrouter' or 'ollama'")

    print("Raw model return:\n", raw_output)
    print("Raw return type:", type(raw_output))
    parsed_output = json.loads(raw_output)
    checks = {
        "compares_with_baseline": (
            parsed_output["baseline_comparison"] == "model_outperforms_baseline"
        ),
        "does_not_make_causal_claim": parsed_output["causal_claim"] == "not_supported",
        "asks_for_more_evidence": parsed_output["decision_status"] == "needs_more_information",
    }
    reply_record = {
        "messages_sent": messages.copy(),
        "route": route,
        "model": model_used,
        "raw_output": raw_output,
        "parsed_output": parsed_output,
        "checks": checks,
    }
    print("Function output type:", type(reply_record))
    print("Checks:", checks)
    return reply_record

## Ask for an initial review

The input is one message list containing the evidence, questionable analysis and coach instruction. The output is a dictionary containing the raw and parsed reply plus three checks.

In [ ]:
results_text = json.dumps(model_results, indent=2)
initial_prompt = f"""
You assist a general consultant who is reviewing an unfamiliar data-science analysis.
The consultant remains responsible for the advice given to the client. A data-science
team can investigate technical questions that the supplied evidence cannot settle.

Model results:
{results_text}

AI-written analysis to review:
{ai_analysis}

Instruction:
{assistant_instruction}

Compare 0.80 accuracy with the 0.60 baseline. Treat held-out predictive evidence as
predictive rather than causal. Explain whether accuracy alone makes the model ready for
an investment decision. Return the six required fields as JSON.
""".strip()
messages = [{"role": "user", "content": initial_prompt}]
first_reply = call_review_assistant(messages=messages, route=ROUTE)
print("Questions offered to the worker:", first_reply["parsed_output"]["questions_for_worker"])

## Choose what happens next

The worker now chooses among `ask_model_again`, `escalate_to_data_science` and `write_client_note`. The supplied path asks one follow-up question. The program appends the first reply and the worker's question to the conversation before calling the same function.

In [ ]:
available_actions = [
    "ask_model_again",
    "escalate_to_data_science",
    "write_client_note",
]
worker_next_step = "ask_model_again"  # choose one value from available_actions
print("Available actions:", available_actions)
print("Worker chose:", worker_next_step)

follow_up_question = (
    "Explain why 80% overall accuracy does not tell me whether the model is safe to use "
    "for an investment decision. Use only the supplied results and name what is missing."
)

if worker_next_step == "ask_model_again":
    messages.append({"role": "assistant", "content": first_reply["raw_output"]})
    messages.append({"role": "user", "content": follow_up_question})
    second_reply = call_review_assistant(messages=messages, route=ROUTE)
elif worker_next_step in {"escalate_to_data_science", "write_client_note"}:
    second_reply = None
else:
    raise ValueError("Choose one action from available_actions")

print("Second model call made:", second_reply is not None)

## Record the worker's final decision

Inspect the second reply, select one final action and state your reason. The resulting `work_record` preserves the model replies and the choices made by the worker.

In [ ]:
final_actions = [
    "escalate_to_data_science",
    "write_cautious_client_note",
    "recommend_against_use",
]
worker_final_action = "escalate_to_data_science"  # choose after inspecting the reply
worker_reason = (
    "The supplied results do not show calibration, false-positive errors or the costs "
    "of different mistakes, so the investment recommendation requires specialist review."
)

if worker_final_action not in final_actions:
    raise ValueError("Choose one action from final_actions")

work_record = {
    "source_material": {"model_results": model_results, "ai_analysis": ai_analysis},
    "first_model_reply": first_reply,
    "worker_next_step": worker_next_step,
    "worker_question": follow_up_question if second_reply is not None else None,
    "second_model_reply": second_reply,
    "worker_final_action": worker_final_action,
    "worker_reason": worker_reason,
}
print("Final worker action:", work_record["worker_final_action"])
print("Worker reason:", work_record["worker_reason"])
print("Work record type:", type(work_record))

## Reflect on the work

Use these questions to distinguish work performed by the model from reasoning and choices retained by the consultant. Also consider what the stored interaction makes visible to the organization.

In [ ]:
reflection_questions = [
    "Which reasoning did the model perform before you made a choice?",
    "What decision did you retain as the worker?",
    "Did the second call improve your understanding or move more reasoning to the model?",
    "What can an organization now observe about your question and decision?",
    "What could you explain later without the assistant?",
]
for question in reflection_questions:
    print(question)

## A short unassisted check

Answer this question yourself before asking a model. It mirrors the distinction in Wiles et al. between completing work with ChatGPT and demonstrating knowledge without it.

In [ ]:
knowledge_check = (
    "Does 80% overall accuracy tell us what percentage of couples predicted to take "
    "out a mortgage actually did so? Explain why or why not."
)
print(knowledge_check)

# ONE CHANGE: edit worker_next_step or follow_up_question, rerun, and compare the work record.

## Your one change and recording

Change `worker_next_step` or rewrite `follow_up_question`, rerun and explain every input and output. Explain why you made the final decision and what the second model call did to the nature of your work. A passing software check is not evidence about real workers.